<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: The original transformer, step by step

**[The original transformer, step by step](https://learning.nextia-ai.com/courses/deep-learning/m08/the-original-transformer/)** · Deep Learning and Transformers Explained · Module 8 (optional), lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build the pieces of the 2017 transformer from their formulas (sinusoidal positions, layer normalization, a post-norm encoder layer), check them against PyTorch, train the paper's recipe on a toy task (reverse a sequence of letters), decode step by step, see the cross-attention, and count the base model's parameters.

| | |
|---|---|
| **Time** | About 40 minutes, including about 1 minute of training |
| **Needs** | No data download and no account. Colab and Kaggle have PyTorch. |
| **You should know** | [Transformer blocks](https://learning.nextia-ai.com/courses/deep-learning/m06/transformer-blocks/) and [The mathematics of attention](https://learning.nextia-ai.com/courses/deep-learning/m08/the-mathematics-of-attention/). |
| **You do not need** | Any earlier notebook of this course. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/the-original-transformer/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M08-L04-the-original-transformer/the-original-transformer-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Sinusoidal positional encoding

The paper's formula: PE(pos, 2i) = sin(pos / 10000^(2i/d_model)) and PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model)). Even columns hold sines, odd columns cosines.

> **Predict.** What are the first 4 numbers of position 1 for d_model = 64? (sin 1, cos 1, then the sine and cosine of 1 / 10000^(2/64).)

> **Your turn.** Write `my_positions(n, d)`, which returns an `(n, d)` tensor. Hint: make the angles `pos / 10000 ** (2i / d)` for 2i = 0, 2, 4, …, then fill `pe[:, 0::2]` and `pe[:, 1::2]`.

In [ ]:
import math

def my_positions(n, d):
    pos = torch.arange(n, dtype=torch.float).unsqueeze(1)    # (n, 1)
    two_i = torch.arange(0, d, 2, dtype=torch.float)         # 2i = 0, 2, ..., d - 2
    angle = pos / torch.pow(10000.0, two_i / d)              # (n, d/2)
    pe = torch.zeros(n, d)
    pe[:, 0::2] = torch.sin(angle)
    pe[:, 1::2] = torch.cos(angle)
    return pe

pe = my_positions(50, 64)
show_shape("pe", pe)
print(pe[1, :4].round(decimals=3))

In [ ]:
expect_hash('pe[1, :4]', [round(float(v), 3) for v in pe[1, :4]] if isinstance(pe, torch.Tensor) else None, 'a225fc37cf0a3e4d')

A shift by k positions is a rotation of each (sine, cosine) pair by the angle ω·k, the same at every position. The cell checks it for k = 3 on columns 10 and 11.

In [ ]:
if isinstance(pe, torch.Tensor):
    k, w = 3, 1 / 10000 ** (10 / 64)
    turn = torch.tensor([[math.cos(w * k), math.sin(w * k)], [-math.sin(w * k), math.cos(w * k)]])
    for pos in (0, 10, 40):
        print(pos, (turn @ pe[pos, 10:12]).round(decimals=4), pe[pos + k, 10:12].round(decimals=4))

## 2. Layer normalization by hand

LayerNorm(x) = γ ⊙ (x − μ) / √(σ² + ε) + β, where μ is the mean of the token's numbers and σ² their variance, **divided by d, not d − 1**. A new layer has γ = 1 and β = 0, and PyTorch uses ε = 10⁻⁵.

> **Your turn.** Compute `by_hand` for x = (1, 2, 3, 6), using only `mean`, `**`, `sqrt` and arithmetic.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, 6.0])
mu = x.mean()
var = ((x - mu) ** 2).mean()
by_hand = (x - mu) / torch.sqrt(var + 1e-5)
print(by_hand.round(decimals=3))
print(torch.nn.LayerNorm(4)(x).detach().round(decimals=3))

In [ ]:
expect('by_hand equals nn.LayerNorm (difference below 1e-5)', isinstance(by_hand, torch.Tensor) and (by_hand - torch.nn.LayerNorm(4)(x).detach()).abs().max().item() < 1e-5, True)

## 3. The model, from the formulas

The next cell defines every part of the paper's model, as in the lesson: `attention` and `MultiHead` (from [The mathematics of attention](https://learning.nextia-ai.com/courses/deep-learning/m08/the-mathematics-of-attention/)), the feed-forward network, the post-norm encoder and decoder layers, the causal mask, and the whole `Transformer` with embedding × √d_model, sinusoidal positions and an output layer tied to the embedding. Read it, then run it.

In [ ]:
import math, random
from torch import nn

def sinusoidal_positions(n, d):
    pos = torch.arange(n, dtype=torch.float).unsqueeze(1)
    i2 = torch.arange(0, d, 2, dtype=torch.float)
    angle = pos / torch.pow(10000.0, i2 / d)
    pe = torch.zeros(n, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(angle), torch.cos(angle)
    return pe

def attention(Q, K, V, mask=None):                    # softmax(Q K^T / sqrt(d_k) + mask) V
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
    if mask is not None:
        scores = scores + mask                        # -inf where a query may not look
    weights = torch.softmax(scores, dim=-1)
    return weights @ V, weights

class MultiHead(nn.Module):                           # h heads of d / h numbers, then W_O
    def __init__(self, d, h):
        super().__init__()
        self.d, self.h = d, h
        self.W_Q, self.W_K, self.W_V, self.W_O = (nn.Linear(d, d) for _ in range(4))
    def split(self, X):                               # (B, n, d) -> (B, h, n, d/h)
        B, n, _ = X.shape
        return X.view(B, n, self.h, self.d // self.h).transpose(1, 2)
    def forward(self, Xq, Xkv, mask=None):            # queries from Xq; keys and values from Xkv
        O, w = attention(self.split(self.W_Q(Xq)), self.split(self.W_K(Xkv)), self.split(self.W_V(Xkv)), mask)
        B, _, n, _ = O.shape
        return self.W_O(O.transpose(1, 2).reshape(B, n, self.d)), w

class FeedForward(nn.Module):                         # max(0, x W1 + b1) W2 + b2
    def __init__(self, d, d_ff):
        super().__init__()
        self.lin1, self.lin2 = nn.Linear(d, d_ff), nn.Linear(d_ff, d)
    def forward(self, x):
        return self.lin2(torch.relu(self.lin1(x)))

class EncoderLayer(nn.Module):                        # post-norm: LayerNorm(x + Dropout(Sublayer(x)))
    def __init__(self, d, h, d_ff, p):
        super().__init__()
        self.self_attn, self.ff = MultiHead(d, h), FeedForward(d, d_ff)
        self.norm1, self.norm2, self.drop = nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(p)
    def forward(self, x, mask):
        x = self.norm1(x + self.drop(self.self_attn(x, x, mask)[0]))
        return self.norm2(x + self.drop(self.ff(x)))

class DecoderLayer(nn.Module):                        # masked self-attention, cross-attention, FFN
    def __init__(self, d, h, d_ff, p):
        super().__init__()
        self.self_attn, self.cross_attn, self.ff = MultiHead(d, h), MultiHead(d, h), FeedForward(d, d_ff)
        self.norm1, self.norm2, self.norm3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.drop = nn.Dropout(p)
    def forward(self, y, memory, tgt_mask, mem_mask, keep=None):
        y = self.norm1(y + self.drop(self.self_attn(y, y, tgt_mask)[0]))
        c, w = self.cross_attn(y, memory, mem_mask)   # queries from the decoder, keys/values from the encoder
        if keep is not None:
            keep.append(w)
        y = self.norm2(y + self.drop(c))
        return self.norm3(y + self.drop(self.ff(y)))

def causal_mask(n):
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)

class Transformer(nn.Module):
    def __init__(self, vocab, d=64, h=4, d_ff=256, n_layers=2, p=0.1, max_len=64, pad=0):
        super().__init__()
        self.d, self.pad = d, pad
        self.embed = nn.Embedding(vocab, d)           # shared: source, target and output
        nn.init.normal_(self.embed.weight, 0, d ** -0.5)
        self.register_buffer("pe", sinusoidal_positions(max_len, d))
        self.drop = nn.Dropout(p)
        self.encoder = nn.ModuleList(EncoderLayer(d, h, d_ff, p) for _ in range(n_layers))
        self.decoder = nn.ModuleList(DecoderLayer(d, h, d_ff, p) for _ in range(n_layers))
    def embed_tokens(self, ids):                      # embedding x sqrt(d_model) + positions
        return self.drop(self.embed(ids) * math.sqrt(self.d) + self.pe[:ids.shape[1]])
    def key_mask(self, ids):                          # -inf on padding keys
        return torch.zeros(ids.shape).masked_fill(ids == self.pad, float("-inf"))[:, None, None, :]
    def encode(self, src):
        x = self.embed_tokens(src)
        for layer in self.encoder:
            x = layer(x, self.key_mask(src))
        return x
    def decode(self, tgt_in, memory, src, keep=None):
        y = self.embed_tokens(tgt_in)
        mask = causal_mask(tgt_in.shape[1]) + self.key_mask(tgt_in)
        for layer in self.decoder:
            y = layer(y, memory, mask, self.key_mask(src), keep)
        return y @ self.embed.weight.T                # output layer tied to the embedding
    def forward(self, src, tgt_in):
        return self.decode(tgt_in, self.encode(src), src)

print("model classes ready")

### Check one encoder layer against PyTorch

PyTorch's `nn.TransformerEncoderLayer(norm_first=False)` is the paper's post-norm layer. Copy the weights of our `EncoderLayer` into it and compare the outputs.

> **Your turn.** Complete `copy_weights`: PyTorch keeps W_Q, W_K and W_V stacked in `self_attn.in_proj_weight` (and their biases in `in_proj_bias`), W_O in `self_attn.out_proj`, the FFN in `linear1` and `linear2`, and the norms in `norm1` and `norm2`. Then compute `diff`, the largest absolute difference of the two outputs in eval mode.

In [ ]:
torch.manual_seed(0)
mine = EncoderLayer(64, 4, 256, 0.1).eval()
ref = nn.TransformerEncoderLayer(64, 4, 256, dropout=0.1, activation="relu", batch_first=True, norm_first=False).eval()

def copy_weights(mine, ref):
    with torch.no_grad():
        a = mine.self_attn
        ref.self_attn.in_proj_weight.copy_(torch.cat([a.W_Q.weight, a.W_K.weight, a.W_V.weight]))
        ref.self_attn.in_proj_bias.copy_(torch.cat([a.W_Q.bias, a.W_K.bias, a.W_V.bias]))
        ref.self_attn.out_proj.weight.copy_(a.W_O.weight)
        ref.self_attn.out_proj.bias.copy_(a.W_O.bias)
        for m, r in ((mine.ff.lin1, ref.linear1), (mine.ff.lin2, ref.linear2), (mine.norm1, ref.norm1), (mine.norm2, ref.norm2)):
            r.weight.copy_(m.weight)
            r.bias.copy_(m.bias)

copy_weights(mine, ref)
x = torch.randn(2, 7, 64)
with torch.no_grad():
    diff = (mine(x, None) - ref(x)).abs().max().item()
print(f"largest difference {diff:.1e}")

In [ ]:
expect('the encoder layer matches PyTorch (difference below 1e-5)', isinstance(diff, float) and diff < 1e-5, True)

### Cross-attention shapes

> **Predict.** The decoder has 6 positions and the source 5 tokens. What shape do the cross-attention weights of one decoder layer have, with a batch of 1 and 4 heads?

In [ ]:
torch.manual_seed(0)
layer = DecoderLayer(64, 4, 256, 0.1).eval()
memory, y = torch.randn(1, 5, 64), torch.randn(1, 6, 64)
keep = []
with torch.no_grad():
    out = layer(y, memory, causal_mask(6), None, keep)
show_shape("output", out)
show_shape("cross w", keep[0])

In [ ]:
expect('the shape of the cross-attention weights', tuple(keep[0].shape), (1, 4, 6, 5))

## 4. The learning-rate schedule

lrate = d_model^−0.5 · min(step^−0.5, step · warmup^−1.5).

> **Your turn.** Write `lrate(step, d_model, warmup)` and compute `peak`, the rate of the paper's base model (d_model 512, warmup 4,000) at step 4,000.

In [ ]:
def lrate(step, d_model, warmup):
    return d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)

peak = lrate(4000, 512, 4000)
print(f"{peak:.3e}", f"{lrate(1000, 512, 4000):.3e}", f"{lrate(100000, 512, 4000):.3e}")

In [ ]:
expect('the peak learning rate (rounded to 4 significant digits)', float(f'{peak:.4g}') if isinstance(peak, float) else None, 0.0006988)

## 5. The toy task: reverse a sequence

Source "c a b e d", decoder input `<s> d e b a c`, target `d e b a c </s>`. The next cell makes random batches and defines greedy decoding and the exact-match score (a sequence counts only if every token is right).

In [ ]:
PAD, BOS, EOS = 0, 1, 2
ITOS = ["<pad>", "<s>", "</s>"] + list("abcdefghij")
STOI = {s: i for i, s in enumerate(ITOS)}

def batch(n, gen, lo=3, hi=8):
    """n random sequences of lo-hi letters: source, decoder input (<s> + reversed), target (reversed + </s>)."""
    lens = torch.randint(lo, hi + 1, (n,), generator=gen)
    L = int(lens.max())
    src, tgt_in, tgt_out = (torch.zeros(n, L + k, dtype=torch.long) for k in (0, 1, 1))
    for i, l in enumerate(lens.tolist()):
        s = torch.randint(3, 13, (l,), generator=gen)
        src[i, :l] = s
        tgt_in[i, 0], tgt_in[i, 1:l + 1] = BOS, s.flip(0)
        tgt_out[i, :l], tgt_out[i, l] = s.flip(0), EOS
    return src, tgt_in, tgt_out

@torch.no_grad()
def greedy(model, src, max_len=12):
    model.eval()
    memory = model.encode(src)
    ys = torch.full((src.shape[0], 1), BOS)
    for _ in range(max_len):
        nxt = model.decode(ys, memory, src)[:, -1].argmax(-1, keepdim=True)
        ys = torch.cat([ys, nxt], 1)
        if (nxt == EOS).all():
            break
    return ys

def exact_match(model, src, tgt_out):
    ys, ok = greedy(model, src, src.shape[1] + 2), 0
    for i in range(len(src)):
        want = [t for t in tgt_out[i].tolist() if t != PAD]
        got = ys[i, 1:].tolist()
        ok += (got[:got.index(EOS) + 1] if EOS in got else got) == want
    return ok / len(src)

src, tgt_in, tgt_out = batch(2, torch.Generator().manual_seed(0))
print("source       ", [[ITOS[i] for i in row] for row in src.tolist()])
print("decoder input", [[ITOS[i] for i in row] for row in tgt_in.tolist()])
print("target       ", [[ITOS[i] for i in row] for row in tgt_out.tolist()])

### Train with the paper's recipe

Adam (β₁ 0.9, β₂ 0.98, ε 10⁻⁹), the warmup schedule (scaled down: warmup 400 for d_model 64), dropout 0.1 and label smoothing 0.1. **This notebook trains 1,500 steps; the lesson trained 3,000** and reached 100% exact match. About 1 minute on a free Colab CPU.

> **Predict.** The smoothed training loss cannot go below 0.537 here. Why? (The lesson explains it in step 6.)

In [ ]:
import time
torch.manual_seed(0); random.seed(0)
gen = torch.Generator().manual_seed(0)
test_src, test_in, test_out = batch(500, torch.Generator().manual_seed(1234))
model = Transformer(vocab=13)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
opt = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
toy_rate = lambda s: 64 ** -0.5 * min(s ** -0.5, s * 400 ** -1.5)   # d_model 64, warmup 400
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: toy_rate(s + 1))
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)
t0 = time.time()
for step in range(1, 1501):
    model.train()
    s, ti, to = batch(64, gen)
    loss = loss_fn(model(s, ti).reshape(-1, 13), to.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    sched.step()
    if step in (1, 500, 1000, 1500):
        print(f"step {step:>5}  lr {opt.param_groups[0]['lr']:.2e}  train loss {loss.item():.3f}"
              f"  exact match {exact_match(model, test_src, test_out):.1%}  ({time.time() - t0:.0f} s)")
acc = exact_match(model, test_src, test_out)

In [ ]:
expect('exact match of at least 90% on the 500 test sequences', acc >= 0.9, True)

### Greedy decoding, step by step

> **Predict.** At each step, will the best token's probability be close to 0.99 or close to 0.91?

In [ ]:
model.eval()
src_ex = torch.tensor([[STOI[c] for c in "cabed"]])
with torch.no_grad():
    memory = model.encode(src_ex)
    ys = torch.tensor([[BOS]])
    for _ in range(10):
        p = model.decode(ys, memory, src_ex)[0, -1].softmax(-1)
        nxt = int(p.argmax())
        print(f"{' '.join(ITOS[i] for i in ys[0].tolist()):<14} -> {ITOS[nxt]:<5} {p[nxt].item():.3f}")
        ys = torch.cat([ys, torch.tensor([[nxt]])], dim=1)
        if nxt == EOS:
            break
answer = " ".join(ITOS[i] for i in ys[0, 1:].tolist())

In [ ]:
expect('the decoded answer', answer, 'd e b a c </s>')

### Where does the decoder look?

The next cell keeps the cross-attention weights of both decoder layers for the example and draws layer 2, averaged over the 4 heads.

> **Predict.** Which source letter will decoder position 1 read most?

In [ ]:
import matplotlib.pyplot as plt
tgt_ex = torch.tensor([[BOS] + [STOI[c] for c in "debac"]])
keep = []
with torch.no_grad():
    model.decode(tgt_ex, model.encode(src_ex), src_ex, keep)
w2 = keep[1][0].mean(0)                               # layer 2, mean of heads: (6, 5)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(w2, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(5), list("cabed")); ax.set_yticks(range(6), ["<s>", "d", "e", "b", "a", "c"])
ax.set_xlabel("source letter"); ax.set_ylabel("decoder input")
for i in range(6):
    for j in range(5):
        ax.text(j, i, f"{w2[i, j]:.2f}", ha="center", va="center", color="white" if w2[i, j] > 0.5 else "black", fontsize=9)
ax.set_title("Cross-attention, decoder layer 2")
plt.show()
reads = [int(w2[i].argmax()) + 1 for i in range(5)]
print("source position read most by decoder positions 1-5:", reads)

In [ ]:
expect('the anti-diagonal (source positions read by decoder positions 1-5)', reads, [5, 4, 3, 2, 1])

Your weights can differ a little from the lesson's, because this model trained for 1,500 steps instead of 3,000. The pattern is the point: to write the reversed text, each position reads the mirrored source letter.

## 6. How many parameters has the paper's base model?

N = 6 encoder and 6 decoder layers, d_model = 512, d_ff = 2,048, and one shared 37,000 × 512 matrix for the source and target embeddings and the output layer. Attention has 4d² + 4d parameters; the FFN d·d_ff + d_ff + d_ff·d + d; each LayerNorm 2d. An encoder layer has 1 attention and 2 norms; a decoder layer 2 attentions and 3 norms.

> **Your turn.** Compute `total` by formula. Then check it by building the model on PyTorch's `meta` device, which creates the shapes without memory.

In [ ]:
d, d_ff, N, vocab = 512, 2048, 6, 37000
attention_n = 4 * d * d + 4 * d
ffn_n = d * d_ff + d_ff + d_ff * d + d
norm_n = 2 * d
total = vocab * d + N * ((attention_n + ffn_n + 2 * norm_n) + (2 * attention_n + ffn_n + 3 * norm_n))
print(f"{total:,}")

In [ ]:
with torch.device("meta"):
    base = Transformer(37000, d=512, h=8, d_ff=2048, n_layers=6, max_len=8)
print(f"PyTorch counts {sum(p.numel() for p in base.parameters()):,}")

In [ ]:
expect("the base model's parameter count", total, 63082496)

The paper's Table 3 says 65 million, rounded, without the exact vocabulary size. With a separate output matrix, the count would be 82.0 million: the sharing matters.

## Next

You built the 2017 transformer from its formulas and watched it learn to reverse. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/the-original-transformer/) has the step-through explorer, the charts, what changed since 2017 and the knowledge checks.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Classify clothing images](https://learning.nextia-ai.com/courses/deep-learning/m09/classify-clothing-images/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m08/the-original-transformer/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The toy data is generated in the notebook; it has no real people.